[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/surfaces/whitney_forms/whitney_forms.ipynb)

# Whitney Forms

This example computes Whitney forms, the functions that spread values stored on the vertices and edges of a triangle mesh into fields across its triangles. A vertex value spreads as a hat, a ramp over the triangles around its vertex, so that neighbouring triangles meet without a seam. An edge value, the integral of a field along that edge, spreads as a field whose reading along the edge is the same from both of its triangles, even where the field itself jumps across it. Steady electric conduction through an anisotropic sheet puts these fields to work: the material acts on the field inside each triangle, and its response returns to the vertices and edges of the mesh.

A position is modelled as an antivector, so a triangle's signed area is the join of two of its sides, `(sides[0] & sides[1]) / 2`, an intrinsic patch. Dividing by twice that patch turns intrinsic directions into extrinsic forms, which read displacements by the regressive product without a metric; the corner gradients and the edge forms both arise this way. Assembled over the mesh, they become maps from vertex and edge values to fields on the faces, and these maps commute with the exterior derivative. The metric enters only with the material, a conductivity that maps the gradient of a potential to the current it drives; the gradient's adjugate carries that current back to the vertices, and the material pulled back through the gradient is the conductance between them.

In the notation of finite element exterior calculus, the hats and edge forms are the lowest-order Whitney forms, which span the spaces $\mathcal P_1^-\Lambda^0$ and $\mathcal P_1^-\Lambda^1$ of the discrete de Rham complex, and the Whitney maps $W^k$ from $k$-cochains to forms commute with the exterior derivative, $d\,W^k c = W^{k+1}\,d c$, where the $d$ on the right is the coboundary of the mesh.


In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import stack
from numga.algebras import VGA2D as ga
from numga.sparse import SparseExtensor, spdiag
from examples import instantiate
from examples.surfaces.whitney_forms import render, scenarios
from examples.util.animation import save_figure

mesh_module = instantiate("examples.util.mesh", ga)
as_scalar, mv = mesh_module.as_scalar, mesh_module.mv
Scalar = ga.gatype.scalar()                                # Scalar+
Plane = ga.gatype.vector()                                 # Vector+

np.set_printoptions(precision=4, suppress=True)

## Corner hats

Each corner hat rises from zero on the opposite edge to one at its corner. Around a shared vertex, the ramps meet continuously even though their slopes change from face to face. The gradient reads a displacement through `gradient & displacement`.

In [ ]:
n_triangle_subdivisions = 8
n_star_divisions = 2
disk_radius = 1.0

In [ ]:
corners, points = scenarios.single_triangle(n_triangle_subdivisions)  # [corners] Point-, [samples] Point-
star_mesh, star_vertices = scenarios.disk(disk_radius, n_star_divisions)  # Mesh, Point-[vertices]

# The signed patch and opposite edges determine the three affine hat gradients.
sides = corners[[2, 0, 1]] - corners[[1, 2, 0]]                        # [corners] Point-
patch = (sides[0] & sides[1]) / 2                                      # [] Patch-
gradients = sides / (2 * patch)                                        # [corners] Plane+
hats = 1 + (gradients[:, None] & (points[None, :] - corners[:, None]))  # [corners, samples] Scalar+

# On the vertex star, assembling the three corner gradients per face maps vertex values to face gradients.
star_edges = star_mesh.d[0] * star_vertices                # Point-[edges]
star_sides = star_mesh.boundary[2, 1] * star_edges         # [corners] Point-[faces]
star_patches = (star_sides[0] & star_sides[1]) / 2         # Patch-[faces]
star_corner_gradients = star_sides / (2 * star_patches)    # [corners] Plane+[faces]
star_gradient = (spdiag(star_corner_gradients) * star_mesh.incidence[2, 0]).sum(axis=0) * Scalar  # (Plane+ <- Scalar+)[faces <- vertices]
vertex_hat = as_scalar(np.arange(star_mesh.n_vertices) == 0).field()  # Scalar+[vertices]
star_gradients = star_gradient(vertex_hat)                 # Plane+[faces]

save_figure(
    render.draw_hat_0forms(corners, gradients, star_mesh, star_vertices, vertex_hat, star_gradients),
    "whitney_hat_0form",
);

The orange arrows show the constant gradient on each triangle; the blue shading shows the hat's height. The vertex star assembles those local ramps into one continuous hat.

In finite element notation, the corner hats are the barycentric functions $\lambda_i$, with constant differentials $d\lambda_i$ on each triangle.

In [ ]:
print("Triangle area:", patch.norm().to_array())

## Edge forms

An edge reading measures a field along a displacement. The corner gradient `sides / (2 * patch)` and the edge form `(points - corners) / (2 * patch)` share their division by twice the triangle's patch: the gradient divides the opposite edge, the form the displacement from the opposite corner. This makes an edge's own reading independent of the position along it, so adjacent triangles agree there.

In [ ]:
# The edge form reads the displacement from its opposite corner through the signed patch.
edge_forms = (points[None, :] - corners[:, None]) / (2 * patch)        # [corners, samples] Plane+
left_corners, left_points, right_corners, right_points = scenarios.edge_diamond(n_triangle_subdivisions)
diamond_corners = stack((left_corners, right_corners))                 # [faces, corners] Point-
diamond_points = stack((left_points, right_points))                   # [faces, samples] Point-
diamond_sides = diamond_corners[:, [2, 0, 1]] - diamond_corners[:, [1, 2, 0]]  # [faces, corners] Point-
diamond_patches = (diamond_sides[:, 0] & diamond_sides[:, 1]) / 2      # [faces] Patch-
diamond_forms = (diamond_points[:, None] - diamond_corners[:, :, None]) / (2 * diamond_patches[:, None, None])  # [faces, corners, samples] Plane+
left_shared, right_shared = diamond_forms[0, 2], -diamond_forms[1, 2]  # [samples] Plane+, Plane+

shared_edge = left_corners[1] - left_corners[0]                         # [] Point-
shared_midpoint = (left_corners[0] + left_corners[1]) / 2               # [] Point-
seam_forms = (shared_midpoint - diamond_corners[:, 2]) / (2 * diamond_patches)  # [faces] Plane+
left_at_seam, right_at_seam = seam_forms[0], -seam_forms[1]            # [] Plane+, Plane+
save_figure(
    render.draw_edge_1forms(corners, points, edge_forms,
                            left_corners, left_points, left_shared,
                            right_corners, right_points, right_shared),
    "whitney_edge_1form",
);

The orange edge carries unit circulation. In the diamond, the arrows can change across the shared edge while their tangential readings agree. The other two edges of each triangle carry zero tangential reading.

In the differential-form notation of finite element exterior calculus, the edge form is $w_{ij}=\lambda_i\,d\lambda_j-\lambda_j\,d\lambda_i$, with $dw_{ij}=2\,d\lambda_i\wedge d\lambda_j$.

In [ ]:
print("Normal reading (left, right):", (left_at_seam & mv.x.dual()).to_array(), (right_at_seam & mv.x.dual()).to_array())

## Material response

Conductivity maps a potential gradient to a current direction. The gradient's adjugate brings the material's response back to the mesh, so changing the material leaves the interpolation intact. On edges, midpoint quadrature integrates the products of Whitney forms exactly, including on obtuse triangles.

In [ ]:
n_disk_divisions = 9
electrode_threshold = 0.8
fibre_angle = np.pi / 6
principal_conductivities = np.array([[1, 1], [10, 1]])     # [cases, axes]

Assembled over a whole mesh, the corner gradients and edge forms of its faces become maps from vertex and edge values to fields on the faces. They commute with the exterior derivative: the edge forms of a difference of vertex values are its gradient, `whitney1(d0(potential)) == gradient(potential)`, and their curl is the face form of the circulation, `curl(cochain) == whitney2(d1(cochain))`.

In [ ]:
mesh, vertices = scenarios.disk(disk_radius, n_disk_divisions)  # Mesh, Point-[vertices]
d0 = mesh.d[0] * Scalar                                    # (Scalar+ <- Scalar+)[edges <- vertices]
d1 = mesh.d[1] * Scalar                                    # (Scalar+ <- Scalar+)[faces <- edges]
edges = mesh.d[0] * vertices                               # Point-[edges]
face_corners = mesh.incidence[2, 0] * vertices             # [face_corners] Point-[faces]
face_sides = mesh.boundary[2, 1] * edges                   # [face_corners] Point-[faces]
patches = (face_sides[0] & face_sides[1]) / 2              # Patch-[faces]
areas = patches.norm()                                     # Scalar+[faces]
centres = face_corners.sum(axis=0) / 3                     # Point-[faces]

# Each corner hat's gradient is constant across its face; assembling by corner maps
# vertex potentials to piecewise-constant face gradients without any metric.
corner_gradients = face_sides / (2 * patches)              # [face_corners] Plane+[faces]
gradient = (spdiag(corner_gradients) * mesh.incidence[2, 0]).sum(axis=0) * Scalar  # (Plane+ <- Scalar+)[faces <- vertices]

# At each face centroid the three barycentric hats equal 1/3, so the edge 1-form
# reduces to one third of the difference of its endpoints' corner gradients.
centroid_forms = (corner_gradients[[2, 0, 1]] - corner_gradients[[1, 2, 0]]) / 3  # [face_corners] Plane+[faces]
whitney1 = (spdiag(centroid_forms) * mesh.boundary[2, 1]).sum(axis=0) * Scalar  # (Plane+ <- Scalar+)[faces <- edges]

# The exterior derivative of each edge's Whitney 1-form is twice the wedge of its endpoints'
# gradients, matching the constant Whitney 2-form of unit integral over the face patch.
tail_gradients, head_gradients = corner_gradients[[1, 2, 0]], corner_gradients[[2, 0, 1]]  # [face_corners] Plane+[faces], [face_corners] Plane+[faces]
curl = (spdiag(2 * (tail_gradients ^ head_gradients)) * mesh.boundary[2, 1]).sum(axis=0) * Scalar  # (Bivector+ <- Scalar+)[faces <- edges]
whitney2 = spdiag(mv.xy / (mv.xy & patches) * Scalar)      # (Bivector+ <- Scalar+)[faces <- faces]

In [ ]:
fibre = (mv.xy * (-fibre_angle / 2)).exp() >> mv.x.dual()  # [] Point-
quarter_turn = (mv.xy * (-np.pi / 4)).exp()                # [] Rotor+
directions = stack((fibre, quarter_turn >> fibre))         # [axes] Point-
conductivity = (directions * (Plane & directions) * principal_conductivities).sum(axis=-1)  # [cases] Point- <- Plane+

boundary = np.unique(mesh.edges[mesh.boundary_edges])
abscissa = (mv.x & vertices).batch()                       # [vertices] Scalar+
electrodes = boundary[abscissa[boundary].norm() > electrode_threshold]
pinned = SparseExtensor.selection(mv.context, electrodes, mesh.n_vertices)
prescribed = ~pinned * (pinned * (mv.x & vertices))        # Scalar+[vertices]
conductance = gradient.adjugate()(spdiag(areas * conductivity)(gradient))  # [cases] (Amount- <- Scalar+)[vertices <- vertices]
free_vertices = np.setdiff1d(np.arange(mesh.n_vertices), electrodes)
free = ~SparseExtensor.selection(mv.context, free_vertices, mesh.n_vertices) * Scalar  # (Scalar+ <- Scalar+)[vertices <- free]
rhs = -free.adjugate()(conductance(prescribed))            # [cases] Amount-[free]
potential = prescribed + free(free.adjugate()(conductance(free)).solve(rhs))  # [cases] Scalar+[vertices]
current = -conductivity(gradient(potential))               # [cases] Point-[faces]

In [ ]:
# Turning the face current by a quarter turn gives, through the unit isotropic conductivity `Plane.dual()`,
# the target gradient of the streamfunction whose level curves follow the current paths.
laplacian = gradient.adjugate()(spdiag(areas * Plane.dual())(gradient))  # (Amount- <- Scalar+)[vertices <- vertices]
unpinned = ~SparseExtensor.selection(mv.context, np.arange(1, mesh.n_vertices), mesh.n_vertices) * Scalar  # (Scalar+ <- Scalar+)[vertices <- unpinned]
stream_rhs = unpinned.adjugate()(gradient.adjugate()(areas * (quarter_turn >> current)))  # [cases] Amount-[unpinned]
streamfunction = unpinned(unpinned.adjugate()(laplacian(unpinned)).solve(stream_rhs))  # [cases] Scalar+[vertices]

save_figure(
    render.draw_anisotropic_flow(mesh, vertices, centres, potential, streamfunction, current, fibre),
    "whitney_anisotropic_flow",
);

The left disk conducts equally in every direction. In the right disk, conductivity is greater along the tilted fibres, bending the current paths. Colour shows potential, dashed lines show equal potential, and solid lines follow the current.

In the matrix notation of the finite element method, this pullback is the stiffness matrix $K_{pq}=\int_\Omega d\lambda_p\cdot(\sigma\,d\lambda_q)\,dA$; isotropic conductivity gives the cotangent Laplacian.

In [ ]:
print("Centre current (isotropic vs anisotropic):\n", (stack((mv.x, mv.y)) & current.batch()[:, 0, None]).to_array())

Two obtuse triangles sharing a long edge give that edge a negative weight in the circumcentric Hodge star of discrete exterior calculus; its Whitney conductance still stores positive energy.

In [ ]:
# Evaluating the linear Whitney 1-forms at the three edge midpoints integrates
# quadratic products over every triangle exactly.
midpoints = (face_corners[[1, 2, 0]] + face_corners[[2, 0, 1]]) / 2  # [midpoints] Point-[faces]
midpoint_forms = (midpoints[:, None] - face_corners[None, :]) / (2 * patches)  # [midpoints, face_corners] Plane+[faces]
midpoint_whitney1 = (spdiag(midpoint_forms) * mesh.boundary[2, 1]).sum(axis=1) * Scalar  # [midpoints] (Plane+ <- Scalar+)[faces <- edges]
weights = spdiag(((areas / 3) * conductivity)[..., None])  # [cases, midpoints] (Point- <- Plane+)[faces <- faces]
edge_conductances = midpoint_whitney1.adjugate()(weights(midpoint_whitney1)).sum(axis=-1)  # [cases] (Amount- <- Scalar+)[edges <- edges]

obtuse_mesh, obtuse_vertices = scenarios.obtuse_diamond()
obtuse_corners = obtuse_mesh.incidence[2, 0] * obtuse_vertices  # [corners] Point-[faces]
obtuse_sides = obtuse_corners[[2, 0, 1]] - obtuse_corners[[1, 2, 0]]  # [corners] Point-[faces]
obtuse_patches = (obtuse_sides[0] & obtuse_sides[1]) / 2   # Patch-[faces]
obtuse_midpoints = (obtuse_corners[[1, 2, 0]] + obtuse_corners[[2, 0, 1]]) / 2  # [midpoints] Point-[faces]
obtuse_forms = (obtuse_midpoints[:, None] - obtuse_corners[None, :]) / (2 * obtuse_patches)  # [midpoints, corners] Plane+[faces]
obtuse_whitney1 = (spdiag(obtuse_forms) * obtuse_mesh.boundary[2, 1]).sum(axis=1) * Scalar  # [midpoints] (Plane+ <- Scalar+)[faces <- edges]
obtuse_weights = spdiag((obtuse_patches.norm() / 3) * Plane.dual())  # (Point- <- Plane+)[faces <- faces]
obtuse_edge_conductance = obtuse_whitney1.adjugate()(obtuse_weights(obtuse_whitney1)).sum(axis=0)  # (Amount- <- Scalar+)[edges <- edges]
shared_cochain = as_scalar(np.arange(len(obtuse_mesh.edges)) == 0).field()  # Scalar+[edges]
obtuse_energy = -(obtuse_edge_conductance(shared_cochain) & shared_cochain).sites.sum()  # [] Scalar+

In [ ]:
# checks
corner_hats = 1 + (gradients[:, None] & (corners[None, :] - corners[:, None]))
np.testing.assert_allclose(corner_hats.kernel, np.eye(3)[..., None], atol=1e-14)
np.testing.assert_allclose(hats.sum(axis=0).kernel, 1.0, atol=1e-14)
np.testing.assert_allclose(gradients.sum(axis=0).kernel, 0.0, atol=1e-14)
np.testing.assert_allclose((1 + (gradients & (corners.mean() - corners))).kernel, 1 / 3, atol=1e-13)

first, second = [1, 2, 0], [2, 0, 1]
alternating_forms = hats[first] * gradients[second, None] - hats[second] * gradients[first, None]
np.testing.assert_allclose((edge_forms - alternating_forms).kernel, 0.0, atol=1e-14)
midpoints = (corners[first] + corners[second]) / 2
midpoint_readings = (midpoints[None, :] - corners[:, None]) / (2 * patch) & sides[None, :]
np.testing.assert_allclose(midpoint_readings.kernel, np.eye(3)[..., None], atol=1e-14)

np.testing.assert_allclose((left_at_seam & shared_edge).kernel, 1.0, atol=1e-14)
np.testing.assert_allclose((right_at_seam & shared_edge).kernel, 1.0, atol=1e-14)

linear_potential = (mv.x & vertices) * 0.7 - (mv.y & vertices) * 1.1
np.testing.assert_allclose(d1(d0(linear_potential)).kernel, 0.0, atol=1e-14)
np.testing.assert_allclose((whitney1(d0(linear_potential)) - gradient(linear_potential)).kernel, 0.0, atol=1e-14)
test_cochain = d0(linear_potential) + as_scalar(np.sin(np.arange(len(mesh.edges)))).field()
np.testing.assert_allclose((curl(test_cochain) - whitney2(d1(test_cochain))).kernel, 0.0, atol=1e-12)

quadratic_potential = (mv.x & vertices).squared() - 0.5 * (mv.y & vertices)
np.testing.assert_allclose(
    (d0.adjugate()(edge_conductances(d0(quadratic_potential))) - conductance(quadratic_potential)).kernel,
    0.0, atol=1e-13,
)
np.testing.assert_allclose((conductance - conductance.adjugate()).cells.kernel, 0.0, atol=1e-13)
assert obtuse_energy > 0

np.testing.assert_allclose(free.adjugate()(conductance(potential)).kernel, 0.0, atol=1e-11)